<a href="https://colab.research.google.com/github/areesha-del/AI-ML-Hands-on/blob/main/Advanced_Transformers_%26_Hugging_Face_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Advanced Transformers & Hugging Face Implementation**

**Install Libraries**

In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

**Load Dataset**

In [ ]:
dataset = load_dataset("imdb")

df = pd.DataFrame(dataset["train"].shuffle(seed=42).select(range(2000)))

df["id"] = range(len(df))
df = df[["id","label","text"]]

hf_dataset = Dataset.from_pandas(df)

# Train test split
hf_dataset = hf_dataset.train_test_split(test_size=0.2, seed=42)

print(df["label"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

label
1    1000
0    1000
Name: count, dtype: int64


**Metrics Function**

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred.predictions, eval_pred.label_ids

    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

**Training Function**

In [ ]:
def train_model(model_name):

    print(f"\nTraining {model_name}...\n")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    # Tokenization
    def tokenize(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=256
        )

    tokenized_dataset = hf_dataset.map(tokenize, batched=True)

    # Rename label column ⭐
    tokenized_dataset = tokenized_dataset.rename_column("label","labels")

    tokenized_dataset.set_format(
        "torch",
        columns=["input_ids","attention_mask","labels"]
    )

    training_args = TrainingArguments(
    output_dir=f"./{model_name}",
    eval_strategy="epoch",   # ✅ changed (important)
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        compute_metrics=compute_metrics
    )

    trainer.train()

    return trainer.evaluate()

**model 1:Bert**

In [ ]:
bert_results = train_model("bert-base-uncased")
print("BERT:", bert_results)


Training bert-base-uncased...



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.387423,0.311687,0.867500,0.819905,0.920213,0.867168
2,0.266887,0.290163,0.887500,0.882353,0.877660,0.880000
3,0.164735,0.367338,0.882500,0.869110,0.882979,0.875989


BERT: {'eval_loss': 0.3673379421234131, 'eval_accuracy': 0.8825, 'eval_precision': 0.8691099476439791, 'eval_recall': 0.8829787234042553, 'eval_f1': 0.8759894459102903, 'eval_runtime': 6.4769, 'eval_samples_per_second': 61.758, 'eval_steps_per_second': 3.86, 'epoch': 3.0}


**Model 2:Distilbert**

In [ ]:
distilbert_results = train_model("distilbert-base-uncased")
print("DistilBERT:", distilbert_results)



Training distilbert-base-uncased...



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.393415,0.309518,0.880000,0.888889,0.851064,0.869565
2,0.257483,0.292661,0.882500,0.877005,0.872340,0.874667
3,0.202401,0.309866,0.885000,0.869792,0.888298,0.878947


DistilBERT: {'eval_loss': 0.3098659813404083, 'eval_accuracy': 0.885, 'eval_precision': 0.8697916666666666, 'eval_recall': 0.8882978723404256, 'eval_f1': 0.8789473684210526, 'eval_runtime': 3.4759, 'eval_samples_per_second': 115.079, 'eval_steps_per_second': 7.192, 'epoch': 3.0}


**Model 3:Roberta**

In [ ]:
roberta_results = train_model("roberta-base")
print("RoBERTa:", roberta_results)


Training roberta-base...



config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.325622,0.193573,0.930000,0.896040,0.962766,0.928205
2,0.243536,0.228831,0.912500,0.896373,0.920213,0.908136
3,0.196475,0.283255,0.912500,0.896373,0.920213,0.908136


RoBERTa: {'eval_loss': 0.2832551896572113, 'eval_accuracy': 0.9125, 'eval_precision': 0.8963730569948186, 'eval_recall': 0.9202127659574468, 'eval_f1': 0.9081364829396326, 'eval_runtime': 6.2663, 'eval_samples_per_second': 63.834, 'eval_steps_per_second': 3.99, 'epoch': 3.0}


**comparision table**

In [ ]:
comparison = pd.DataFrame({
    "Model": ["BERT", "DistilBERT", "RoBERTa"],
    "Accuracy": [
        bert_results["eval_accuracy"],
        distilbert_results["eval_accuracy"],
        roberta_results["eval_accuracy"],
    ],
    "Precision": [
        bert_results["eval_precision"],
        distilbert_results["eval_precision"],
        roberta_results["eval_precision"],
    ],
    "Recall": [
        bert_results["eval_recall"],
        distilbert_results["eval_recall"],
        roberta_results["eval_recall"],
    ],
    "F1-score": [
        bert_results["eval_f1"],
        distilbert_results["eval_f1"],
        roberta_results["eval_f1"],
    ],
})

print("\nModel Comparison Table")
print(comparison)
comparison.to_csv("model_comparison.csv", index=False)


Model Comparison Table
        Model  Accuracy  Precision    Recall  F1-score
0        BERT    0.8825   0.869110  0.882979  0.875989
1  DistilBERT    0.8850   0.869792  0.888298  0.878947
2     RoBERTa    0.9125   0.896373  0.920213  0.908136
